# 📊 Analyse exploratoire — Trafic passagers Air France

**Objectif** : Explorer les données officielles DGAC pour comprendre la structure du trafic Air France (2010–2024), identifier les tendances, la saisonnalité et les anomalies (Covid).

**Source** : Direction Générale de l'Aviation Civile (DGAC) — data.gouv.fr  
**Contexte** : Ces données sont utilisées pour construire un modèle de prévision pertinent dans un contexte de Revenue Management.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Librairies chargées')

## 1. Chargement et inspection des données

In [ ]:
df = pd.read_csv('../data/trafic_airfrance.csv', parse_dates=['date'])
df['mois']      = df['date'].dt.month
df['annee']     = df['date'].dt.year
df['trimestre'] = df['date'].dt.quarter
df['covid']     = ((df['date'] >= '2020-03-01') & (df['date'] <= '2021-06-01')).astype(int)

print(f'Période : {df["date"].min().date()} a {df["date"].max().date()}')
print(f'Nombre de mois : {len(df)}')
df.head(12)

In [ ]:
print('Valeurs manquantes :')
print(df.isnull().sum())
print()
df[['passagers','vols']].describe().round(0)

## 2. Evolution du trafic (2010-2024)

Trois phases distinctes :
- **2010-2019** : croissance reguliere avec forte saisonnalite estivale
- **2020-2021** : effondrement brutal lie au Covid-19
- **2022-2024** : reprise progressive

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['date'], df['passagers']/1e6, color='steelblue', linewidth=1.5)
ax.fill_between(df['date'], df['passagers']/1e6, alpha=0.1, color='steelblue')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'), alpha=0.2, color='red', label='Periode Covid')
ax.set_ylabel('Passagers (millions)')
ax.set_title('Evolution mensuelle du trafic passagers Air France (2010-2024)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/evolution_trafic.png', dpi=150)
plt.show()

## 3. Saisonnalite

La saisonnalite est un facteur cle en Revenue Management. Juillet est le mois le plus charge, fevrier le plus creux.

In [ ]:
df_hors_covid = df[df['covid'] == 0]
saisonnalite  = df_hors_covid.groupby('mois')['passagers'].mean() / 1e6
mois_labels   = ['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(mois_labels, saisonnalite.values, color='steelblue', alpha=0.8)
bars[saisonnalite.values.argmax()].set_color('tomato')
ax.set_ylabel('Passagers moyens (millions)')
ax.set_title('Saisonnalite mensuelle moyenne - hors periode Covid')
plt.tight_layout()
plt.savefig('../figures/saisonnalite.png', dpi=150)
plt.show()

## 4. Comparaison annuelle

In [ ]:
mois_labels = ['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec']
fig, ax = plt.subplots(figsize=(12, 5))
palette = plt.cm.tab10.colors
for i, annee in enumerate(sorted(df['annee'].unique())):
    data = df[df['annee'] == annee].sort_values('mois')
    style = '--' if annee in [2020, 2021] else '-'
    width = 2.5 if annee in [2019, 2024] else 1.2
    ax.plot(data['mois'], data['passagers']/1e6, label=str(annee),
            linestyle=style, linewidth=width, color=palette[i % len(palette)])
ax.set_xticks(range(1,13))
ax.set_xticklabels(mois_labels)
ax.set_ylabel('Passagers (millions)')
ax.set_title('Comparaison du trafic mensuel par annee')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../figures/comparaison_annuelle.png', dpi=150)
plt.show()

## 5. Reprise post-Covid

Question cle : quand Air France a-t-elle retrouve son niveau pre-Covid ?
Reference : moyenne mensuelle 2017-2019.

In [ ]:
ref  = df[df['annee'].isin([2017,2018,2019])].groupby('mois')['passagers'].mean()
post = df[df['annee'] >= 2021].copy()
post['ref_mois'] = post['mois'].map(ref)
post['taux']     = (post['passagers'] / post['ref_mois'] * 100).round(1)

date_95 = post[post['taux'] >= 95]['date'].min()
print(f'Premier mois a 95% du niveau pre-Covid : {date_95.strftime("%B %Y")}')
print(f'Taux moyen en 2024 : {post[post["annee"]==2024]["taux"].mean():.1f}%')

fig, ax = plt.subplots(figsize=(12, 4))
couleurs = post['taux'].apply(lambda x: 'tomato' if x < 70 else ('orange' if x < 90 else 'steelblue'))
ax.bar(post['date'], post['taux'], color=couleurs, alpha=0.85, width=25)
ax.axhline(100, color='green', linestyle='--', linewidth=1.5, label='Niveau pre-Covid (100%)')
ax.axhline(95,  color='orange', linestyle=':', linewidth=1.2, label='Seuil 95%')
ax.axvline(date_95, color='purple', linestyle='--', linewidth=1.5)
ax.set_ylabel('% du niveau pre-Covid')
ax.set_ylim(0, 115)
ax.set_title('Taux de recuperation post-Covid - Air France')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/reprise_covid.png', dpi=150)
plt.show()

## 6. Correlations et features

On identifie quelles variables sont les plus predictives pour le modele.

In [ ]:
df['lag_12'] = df['passagers'].shift(12)
df['lag_1']  = df['passagers'].shift(1)
df['mm_6']   = df['passagers'].rolling(6).mean().shift(1)

corr = df[['passagers','mois','annee','lag_1','lag_12','mm_6','vols']].dropna().corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Matrice de correlation')
plt.tight_layout()
plt.savefig('../figures/correlations.png', dpi=150)
plt.show()
print('lag_12 et mm_6 sont tres correlees avec passagers -> bonnes features pour XGBoost')

## 7. Conclusions

### Ce qu'on a appris :
1. **Forte saisonnalite** : pic en juillet (+25% vs moyenne), creux en fevrier (-10%)
2. **Tendance haussiere** sur 2010-2019, interrompue par le Covid
3. **Reprise lente** : 3 ans pour atteindre 95% du niveau pre-Covid (mai 2023)
4. **2024 toujours en dessous** : ~91% du niveau pre-Covid
5. **lag_12** est la variable la plus predictive

### Implications Revenue Management :
- Les mois de basse saison offrent des opportunites de stimulation tarifaire
- La saisonnalite est stable et previsible -> modele XGBoost fiable
- L'ecart avec le niveau pre-Covid suggere un potentiel de croissance non encore capture